# Legal Judgment Annotation Dataset Creator

**Purpose:** Process a directory of judgment `.txt` files and create an initial annotation dataset in Excel format.

**Output columns (exact order):**

| # | Column | Populated? |
|---|--------|------------|
| 1 | `case_id` | Filename (without extension) |
| 2 | `language` | Auto-detected |
| 3 | `case_type` | Blank (for annotation) |
| 4 | `judgment_text` | Full text from file |
| 5 | `subject` | Blank (for annotation) |
| 6 | `object` | Blank (for annotation) |
| 7 | `objective_aspect` | Blank (for annotation) |
| 8 | `subjective_aspect` | Blank (for annotation) |
| 9 | `legal_provision` | Blank (for annotation) |
| 10 | `reasoning` | Blank (for annotation) |

## 1. Install Dependencies

In [3]:
# Uncomment and run if packages are not installed
!pip install pandas openpyxl langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 6.4 MB/s  0:00:00eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993332 sha256=3a6475a2db29c5de595ab9f4b3989c309ac32797f64bc2d3c1bd71baecfdcece
  Stored in directory: /mnt/Data/yashv7523/.cache/pip/wheels/d1/c1/d9/7e068de779d863bc8f8fc9467d85e25cfe47fa5051fff1a1bb
Successfully built langdetect
    torch (>=1.7.*)
           ~~~~~~^


## 2. Imports

In [1]:
import zipfile
import os

#zip_file = "Assamese_OCR.zip"
extract_folder = "Chhattisgarh"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("Extraction completed!")

Extraction completed!


In [ ]:
extract_folder = "Chhattisgarh"

In [1]:
import os
import sys
import re
import unicodedata
from pathlib import Path
from typing import Optional

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
from langdetect import detect, DetectorFactory

# Make langdetect deterministic
DetectorFactory.seed = 0

print("All imports successful.")

/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/mnt/Data/yashv7523/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


All imports successful.


In [4]:
import os

print("Current Working Directory:")
print(os.getcwd())

Current Working Directory:
/mnt/Data/yashv7523/Deepak/INDILEX


## 3. Configuration

In [3]:
# ============================================================
# SET YOUR PATHS HERE
# ============================================================

INPUT_DIR = "/mnt/Data/yashv7523/Deepak/INDILEX/Chhattisgarh"      # Folder with .txt judgment files
OUTPUT_FILE = "Chhattisgarh_dataset.xlsx"             # Output Excel filename

# Hidden/system files to ignore
IGNORED_FILES = {
    ".DS_Store", "Thumbs.db", "desktop.ini",
    ".gitkeep", ".gitignore", "__MACOSX"
}

# Columns in exact required order
COLUMNS = [
    "case_id",
    "language",
    "case_type",
    "judgment_text",
    "subject",
    "object",
    "objective_aspect",
    "subjective_aspect",
    "legal_provision",
    "reasoning",
]

# Columns that must remain blank (for manual annotation)
BLANK_COLUMNS = [
    "case_type", "subject", "object",
    "objective_aspect", "subjective_aspect",
    "legal_provision", "reasoning",
]

print(f"Input directory : {INPUT_DIR}")
print(f"Output file     : {OUTPUT_FILE}")
print(f"Columns ({len(COLUMNS)}): {COLUMNS}")

Input directory : /mnt/Data/yashv7523/Deepak/INDILEX/Chhattisgarh
Output file     : Chhattisgarh_dataset.xlsx
Columns (10): ['case_id', 'language', 'case_type', 'judgment_text', 'subject', 'object', 'objective_aspect', 'subjective_aspect', 'legal_provision', 'reasoning']


## 4. Helper Functions

In [4]:
def is_valid_file(filename: str) -> bool:
    """
    Check if a file should be processed.
    Skips hidden files, system files, and non-.txt files.
    """
    if filename.startswith("."):
        return False
    if filename in IGNORED_FILES:
        return False
    if not filename.lower().endswith(".txt"):
        return False
    return True


def read_file_safe(filepath: str) -> Optional[str]:
    """
    Read a text file with multiple encoding fallbacks.
    Returns the file content or None if all encodings fail.
    """
    encodings = ["utf-8", "utf-8-sig", "utf-16", "latin-1", "cp1252"]
    for enc in encodings:
        try:
            with open(filepath, "r", encoding=enc) as f:
                return f.read()
        except (UnicodeDecodeError, UnicodeError):
            continue
        except Exception:
            return None
    return None


def detect_script(text: str) -> str:
    """
    Detect the dominant Unicode script in the text.
    Returns the script name (e.g., 'BENGALI', 'DEVANAGARI', 'LATIN').
    """
    script_counts = {}
    sample = text[:5000]  # Sample first 5000 chars for speed
    for ch in sample:
        if ch.isalpha():
            try:
                name = unicodedata.name(ch, "")
                script = name.split()[0] if name else "UNKNOWN"
                script_counts[script] = script_counts.get(script, 0) + 1
            except ValueError:
                pass
    if not script_counts:
        return "UNKNOWN"
    return max(script_counts, key=script_counts.get)


def detect_language(text: str, filename: str) -> str:
    """
    Detect the language of a judgment using a three-tier approach:
    
    1. Filename suffix (most reliable for this project)
       e.g., WPC_1056_1999_As.txt → 'As' → Assamese
    2. Unicode script detection (distinguishes script families)
    3. langdetect library (fallback for Latin/Devanagari ambiguity)
    
    Returns a human-readable language name.
    """
    # --- Tier 1: Filename-based language code ---
    # Common pattern: ..._{LangCode}.txt
    lang_code_map = {
        "as": "Assamese", "bn": "Bengali", "en": "English",
        "hi": "Hindi", "gu": "Gujarati", "kn": "Kannada",
        "ml": "Malayalam", "mr": "Marathi", "or": "Odia",
        "pa": "Punjabi", "ta": "Tamil", "te": "Telugu",
        "ur": "Urdu",
    }
    
    stem = Path(filename).stem  # filename without extension
    # Check last segment after underscore or hyphen
    parts = re.split(r"[_\-]", stem)
    if parts:
        suffix = parts[-1].lower()
        if suffix in lang_code_map:
            return lang_code_map[suffix]
    
    # --- Tier 2: Unicode script detection ---
    script = detect_script(text)
    script_lang_map = {
        "BENGALI": "Bengali/Assamese",  # Same script, refine below
        "DEVANAGARI": "Hindi",
        "GUJARATI": "Gujarati",
        "GURMUKHI": "Punjabi",
        "KANNADA": "Kannada",
        "MALAYALAM": "Malayalam",
        "ORIYA": "Odia",
        "TAMIL": "Tamil",
        "TELUGU": "Telugu",
        "ARABIC": "Urdu",
    }
    
    if script in script_lang_map:
        return script_lang_map[script]
    
    # --- Tier 3: langdetect fallback (mostly for Latin-script texts) ---
    try:
        clean_text = re.sub(r"---\s*Page\s+\d+\s*---", "", text)
        clean_text = re.sub(r"\f", "", clean_text).strip()
        if len(clean_text) > 50:
            code = detect(clean_text[:3000])
            langdetect_map = {
                "en": "English", "hi": "Hindi", "bn": "Bengali",
                "mr": "Marathi", "gu": "Gujarati", "ta": "Tamil",
                "te": "Telugu", "kn": "Kannada", "ml": "Malayalam",
                "pa": "Punjabi", "ur": "Urdu", "or": "Odia",
            }
            return langdetect_map.get(code, code.upper())
    except Exception:
        pass
    
    return "Unknown"


# Quick test
print("Helper functions loaded.")

Helper functions loaded.


## 5. Scan Input Directory

In [5]:
# Discover all valid .txt files
all_entries = sorted(os.listdir(INPUT_DIR))
valid_files = [f for f in all_entries if is_valid_file(f)]
skipped_files = [f for f in all_entries if not is_valid_file(f) and f != "." and f != ".."]

print(f"Total entries in directory : {len(all_entries)}")
print(f"Valid .txt files           : {len(valid_files)}")
if skipped_files:
    print(f"Skipped (non-txt/hidden)   : {len(skipped_files)} → {skipped_files[:5]}")

# Preview first 5 filenames
print(f"\nFirst 5 files:")
for f in valid_files[:5]:
    print(f"  {f}")

Total entries in directory : 201
Valid .txt files           : 201

First 5 files:
  ACQ A350_22(12.04.23).txt
  ARB A5_17(03.05.23).txt
  ARBR27_23(24.11.23).txt
  CR27_19(11.08.23).txt
  CRA1021_16(23.11.23).txt


## 6. Process All Judgment Files

In [6]:
records = []       # List of dicts, one per judgment
error_log = []     # Files that could not be read
total = len(valid_files)

for idx, filename in enumerate(valid_files, start=1):
    filepath = os.path.join(INPUT_DIR, filename)
    
    # --- Read file ---
    text = read_file_safe(filepath)
    if text is None:
        error_log.append(filename)
        print(f"  [SKIP] Could not read: {filename}")
        continue
    
    # --- Extract case_id (filename without extension) ---
    case_id = Path(filename).stem
    
    # --- Detect language ---
    language = detect_language(text, filename)
    
    # --- Build record ---
    record = {
        "case_id": case_id,
        "language": language,
        "case_type": "",            # Blank for annotation
        "judgment_text": text,       # Full text preserved as-is
        "subject": "",              # Blank for annotation
        "object": "",               # Blank for annotation
        "objective_aspect": "",     # Blank for annotation
        "subjective_aspect": "",    # Blank for annotation
        "legal_provision": "",      # Blank for annotation
        "reasoning": "",            # Blank for annotation
    }
    records.append(record)
    
    # --- Progress ---
    if idx % 25 == 0 or idx == total:
        print(f"  Processed {idx}/{total} files  |  Language: {language}  |  {case_id}")

print(f"\n{'='*60}")
print(f"Successfully processed : {len(records)}/{total}")
if error_log:
    print(f"Failed files ({len(error_log)}):")
    for ef in error_log:
        print(f"  ✗ {ef}")
else:
    print("No errors — all files read successfully.")

  Processed 25/201 files  |  Language: Hindi  |  CRA387_21(21.08.23)
  Processed 50/201 files  |  Language: Hindi  |  CRA937_01(13.04.23)
  Processed 75/201 files  |  Language: Hindi  |  CRR547_19(10.03.23)
  Processed 100/201 files  |  Language: Hindi  |  MA11_19(01.12.23)
  Processed 125/201 files  |  Language: Hindi  |  WA65_23(19.04.23)
  Processed 150/201 files  |  Language: Hindi  |  WP(CR)230_18(18.07.23)
  Processed 175/201 files  |  Language: Hindi  |  WP(S)2530_21(10.03.23)
  Processed 200/201 files  |  Language: Hindi  |  WP(S)8817_22(28.11.23)
  Processed 201/201 files  |  Language: Hindi  |  WP(S)947_16(06.12.23)

Successfully processed : 201/201
No errors — all files read successfully.


## 7. Build DataFrame

In [7]:
# Create DataFrame with exact column order
df = pd.DataFrame(records, columns=COLUMNS)

print(f"DataFrame shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nLanguage distribution:")
print(df["language"].value_counts().to_string())
print(f"\nSample case_ids:")
print(df["case_id"].head(10).to_string())
print(f"\nAvg judgment text length: {df['judgment_text'].str.len().mean():.0f} chars")
print(f"Min: {df['judgment_text'].str.len().min()} | Max: {df['judgment_text'].str.len().max()}")

DataFrame shape: 201 rows × 10 columns
Columns: ['case_id', 'language', 'case_type', 'judgment_text', 'subject', 'object', 'objective_aspect', 'subjective_aspect', 'legal_provision', 'reasoning']

Language distribution:
language
Hindi    201

Sample case_ids:
0    ACQ A350_22(12.04.23)
1      ARB A5_17(03.05.23)
2      ARBR27_23(24.11.23)
3        CR27_19(11.08.23)
4     CRA1021_16(23.11.23)
5      CRA103_20(26.06.23)
6     CRA1099_13(05.04.23)
7     CRA1115_13(05.05.23)
8     CRA1120_02(28.02.23)
9     CRA1139_14(14.08.23)

Avg judgment text length: 20254 chars
Min: 2805 | Max: 86962


## 8. Save to Formatted Excel

In [9]:
def save_legal_dataset(df: pd.DataFrame, output_path: str) -> None:
    """
    Save the DataFrame to a professionally formatted Excel file.
    - No index column
    - Proper column widths
    - Header styling
    - Text wrapping for judgment_text
    - Preserves line breaks in judgment text
    """
    wb = Workbook()
    ws = wb.active
    ws.title = "Legal Dataset"
    
    # ---- Styles ----
    header_font = Font(name="Arial", bold=True, color="FFFFFF", size=11)
    header_fill = PatternFill("solid", fgColor="1F4E79")
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    
    cell_font = Font(name="Arial", size=10)
    cell_align = Alignment(vertical="top", wrap_text=True)
    text_align = Alignment(vertical="top", wrap_text=True)  # For judgment_text
    
    thin_border = Border(
        left=Side(style="thin", color="B0B0B0"),
        right=Side(style="thin", color="B0B0B0"),
        top=Side(style="thin", color="B0B0B0"),
        bottom=Side(style="thin", color="B0B0B0"),
    )
    
    # Column widths (tuned for annotation workflow)
    col_widths = {
        "case_id": 30,
        "language": 14,
        "case_type": 16,
        "judgment_text": 80,
        "subject": 25,
        "object": 25,
        "objective_aspect": 25,
        "subjective_aspect": 25,
        "legal_provision": 25,
        "reasoning": 30,
    }
    
    columns = list(df.columns)
    
    # ---- Write Header Row ----
    for col_idx, col_name in enumerate(columns, 1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_align
        cell.border = thin_border
    
    # ---- Write Data Rows ----
    total_rows = len(df)
    for row_idx, (_, row) in enumerate(df.iterrows(), 2):
        for col_idx, col_name in enumerate(columns, 1):
            value = row[col_name]
            # Ensure blank columns are truly empty strings
            if col_name in BLANK_COLUMNS:
                value = ""
            elif pd.isna(value):
                value = ""
            else:
                value = str(value)
            
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = cell_font
            cell.border = thin_border
            
            if col_name == "judgment_text":
                cell.alignment = text_align
            else:
                cell.alignment = cell_align
        
        # Progress for large datasets
        if (row_idx - 1) % 50 == 0 or (row_idx - 1) == total_rows:
            print(f"  Writing row {row_idx - 1}/{total_rows}...")
    
    # ---- Set Column Widths ----
    for col_idx, col_name in enumerate(columns, 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = col_widths.get(col_name, 20)
    
    # ---- Freeze Header Row ----
    ws.freeze_panes = "A2"
    
    # ---- Auto-filter ----
    ws.auto_filter.ref = ws.dimensions
    
    # ---- Header Row Height ----
    ws.row_dimensions[1].height = 28
    
    # ---- Save ----
    wb.save(output_path)
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"\n{'='*60}")
    print(f"Excel saved: {output_path}")
    print(f"Rows: {total_rows} | Columns: {len(columns)} | Size: {file_size_mb:.2f} MB")


# Save the dataset
save_legal_dataset(df, OUTPUT_FILE)

  Writing row 50/201...
  Writing row 100/201...
  Writing row 150/201...
  Writing row 200/201...
  Writing row 201/201...

Excel saved: Chhattisgarh_dataset.xlsx
Rows: 201 | Columns: 10 | Size: 2.09 MB


In [10]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# Clean the judgment_text column of any illegal Excel characters (like \f)
df['judgment_text'] = df['judgment_text'].apply(lambda x: ILLEGAL_CHARACTERS_RE.sub('', str(x)))

# Now save the dataset
save_legal_dataset(df, OUTPUT_FILE)

  Writing row 50/201...
  Writing row 100/201...
  Writing row 150/201...
  Writing row 200/201...
  Writing row 201/201...

Excel saved: Chhattisgarh_dataset.xlsx
Rows: 201 | Columns: 10 | Size: 2.09 MB


## 9. Validation

In [11]:
# Re-read the saved Excel to verify ¶
1
# Re-read the saved Excel to verify correctness
2
df_verify = pd.read_excel(OUTPUT_FILE, engine="openpyxl")
3
​
4
print("=== VALIDATION REPORT ===")correctness
df_verify = pd.read_excel(OUTPUT_FILE, engine="openpyxl")

print("=== VALIDATION REPORT ===")
print(f"\n1. Column check:")
expected = COLUMNS
actual = list(df_verify.columns)
if actual == expected:
    print(f"   ✓ All {len(expected)} columns present in correct order")
else:
    print(f"   ✗ Column mismatch!")
    print(f"     Expected: {expected}")
    print(f"     Got:      {actual}")

print(f"\n2. Row count:")
print(f"   ✓ {len(df_verify)} judgments in Excel")

print(f"\n3. Blank columns check:")
for col in BLANK_COLUMNS:
    non_empty = df_verify[col].dropna().astype(str).str.strip().ne("").sum()
    status = "✓" if non_empty == 0 else "✗"
    print(f"   {status} {col}: {'all blank' if non_empty == 0 else f'{non_empty} non-blank!'}")

print(f"\n4. Populated columns check:")
for col in ["case_id", "language", "judgment_text"]:
    filled = df_verify[col].dropna().astype(str).str.strip().ne("").sum()
    pct = filled / len(df_verify) * 100
    status = "✓" if pct == 100 else "⚠"
    print(f"   {status} {col}: {filled}/{len(df_verify)} filled ({pct:.1f}%)")

print(f"\n5. No extra index column:")
has_unnamed = any("Unnamed" in str(c) for c in df_verify.columns)
print(f"   {'✗ Found unnamed column!' if has_unnamed else '✓ Clean — no index column'}")

print(f"\n6. Language distribution:")
print(df_verify["language"].value_counts().to_string())

print(f"\n7. Sample rows:")
print(df_verify[["case_id", "language"]].head(10).to_string())

SyntaxError: invalid non-printable character U+200B (3887770438.py, line 7)

## 10. Error Log

In [ ]:
if error_log:
    print(f"The following {len(error_log)} file(s) could not be read and were skipped:")
    for i, fname in enumerate(error_log, 1):
        print(f"  {i}. {fname}")
else:
    print("No errors. All files processed successfully.")

---
## Notes

- **Line breaks preserved**: `openpyxl` writes `\n` as Excel line breaks; enable "Wrap Text" in Excel to see them.
- **Language detection**: Uses filename suffix (e.g., `_As` → Assamese) as the primary signal. Falls back to Unicode script analysis, then `langdetect`.
- **Assamese vs Bengali**: Both use the Bengali script. The filename suffix `_As` is used to distinguish Assamese from Bengali.
- **Scalability**: Tested with 211 files. For 5000+ files, the same pipeline works — progress is printed every 25 files.
- **Encoding**: Tries UTF-8 first, then UTF-8-BOM, UTF-16, Latin-1, CP1252.